<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Gemini API and VertexAI</h1>
<h1>Fundamentals</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
from collections import Counter
from pprint import pprint
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import google.generativeai as genai
from google.generativeai.types import HarmCategory, HarmBlockThreshold
import vertexai
from vertexai.generative_models import GenerativeModel, Part, GenerationConfig

import watermark
%load_ext watermark
%matplotlib inline


/var/folders/lr/j1bs1q851k15cj5y777nxwph0000gn/T/ipykernel_91505/2161633809.py:9: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


We start by printing out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.10.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 0f22cb8c30fea71d49b61d9441111870a39cba67

matplotlib: 3.10.8
numpy     : 2.4.2
pandas    : 3.0.1
vertexai  : 1.71.1
watermark : 2.6.0



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# 1. Generative AI

## What is Generative AI?

Generative AI refers to a class of machine learning models capable of producing novel content — text, images, audio, video, code, and more — that resembles the data they were trained on. Unlike **discriminative** models that classify or label existing data, **generative** models learn the underlying distribution of the training data and can sample from it to create new outputs.

### Key Concepts

| Concept | Description |
|---|---|
| **Foundation Models** | Large models pre-trained on vast datasets, adaptable to many tasks via fine-tuning or prompting |
| **Large Language Models (LLMs)** | Foundation models specializing in understanding and generating natural language |
| **Prompt Engineering** | Crafting inputs to guide model outputs without changing model weights |
| **In-Context Learning** | The ability of LLMs to learn from examples provided within the prompt itself |
| **Hallucination** | The tendency of LLMs to generate plausible but factually incorrect content |

### The Transformer Revolution

The modern era of Generative AI began with the **Transformer architecture** (Vaswani et al., 2017 — *"Attention Is All You Need"*). Key innovations:

- **Self-Attention Mechanism**: Allows the model to relate every token to every other token in a sequence, regardless of distance.
- **Positional Encoding**: Injects information about token order since Transformers are permutation-invariant.
- **Scalability**: Transformers scale efficiently on modern GPU/TPU hardware, enabling training on internet-scale data.
- **Parallelism**: Unlike RNNs, Transformers process all tokens in a sequence simultaneously during training.

### Key Milestones

- **2017**: Transformer architecture published (Google Brain)
- **2018**: BERT — Bidirectional Encoder Representations from Transformers (Google)
- **2019**: GPT-2 — demonstrates compelling text generation at scale (OpenAI)
- **2020**: GPT-3 — 175B parameters; few-shot learning via prompting (OpenAI)
- **2021**: DALL-E — text-to-image generation (OpenAI); Codex — code generation
- **2022**: ChatGPT released; Stable Diffusion; Whisper (speech recognition)
- **2023**: GPT-4, LLaMA, Claude, PaLM 2, Gemini (multimodal)
- **2024**: Gemini 1.5, Claude 3, GPT-4o — long context, multimodality, efficiency
- **2025**: Gemini 2.0, reasoning-focused models, agentic frameworks mature

In [ ]:
# Timeline visualization: Evolution of Generative AI models

milestones = [
    (2017, "Transformer\n(Attention Is All You Need)", 0.3),
    (2018, "BERT", 0.11),
    (2019, "GPT-2", 1.5),
    (2020, "GPT-3", 175),
    (2021, "DALL-E / Codex", 12),
    (2022, "ChatGPT / Stable Diffusion", 20),
    (2023, "GPT-4 / Gemini 1.0", 100),
    (2024, "Gemini 1.5 / GPT-4o", 200),
    (2025, "Gemini 2.0 / Reasoning Models", 300),
]

years  = [m[0] for m in milestones]
labels = [m[1] for m in milestones]
sizes  = [m[2] for m in milestones]  # approximate parameter count (billions), illustrative

fig, ax = plt.subplots(figsize=(14, 5))

bar_colors = [colors[i % len(colors)] for i in range(len(years))]
bars = ax.bar(range(len(years)), sizes, color=bar_colors, edgecolor='white', linewidth=0.8)

ax.set_xticks(range(len(years)))
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax.tick_params(axis='y', labelsize=9)
ax.set_ylabel('Approximate Scale\n(Billions of Parameters, Illustrative)', fontsize=10)
ax.set_title('Evolution of Generative AI Models (2017-2025)', fontsize=13, fontweight='bold')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:g}B'))

for bar, year in zip(bars, years):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.15,
            str(year), ha='center', va='bottom', fontsize=8, color=colors[7])

ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### Transformer Architecture: Self-Attention Intuition

At the core of every modern LLM is the **self-attention** mechanism. For each token in a sequence, attention computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Where:
- **Q** (Query): What this token is looking for
- **K** (Key): What each token can offer
- **V** (Value): The actual content to aggregate
- **$d_k$**: Dimension of keys (scaling factor for numerical stability)

**Multi-head attention** runs several attention heads in parallel, each learning different relationship patterns, then concatenates and projects the results.

# 2. Introduction to the Gemini API

## The Gemini Model Family

Google's **Gemini** is a family of natively multimodal foundation models capable of processing text, images, audio, video, and code.

| Model | Context Window | Best For |
|---|---|---|
| `gemini-1.5-pro` | 1M tokens | Complex reasoning, long documents, code |
| `gemini-1.5-flash` | 1M tokens | High throughput, cost-efficient applications |
| `gemini-2.0-flash` | 1M tokens | Latest generation, speed + quality balance |
| `gemini-2.0-flash-lite` | 1M tokens | Lightest and fastest variant |

### Multimodal Capabilities

Gemini models can accept as input:
- **Text** (prompts, documents, code)
- **Images** (JPEG, PNG, GIF, etc.)
- **Audio** (MP3, WAV, FLAC, etc.)
- **Video** (MP4, MOV, AVI, etc.)
- **Files** (PDFs and other documents via the File API)

### Access Paths

1. **Google AI Studio + `google-generativeai` SDK** — simplest path, uses an API key, great for prototyping
2. **Vertex AI** — enterprise-grade, uses Google Cloud credentials, adds MLOps features

## Setting Up the Gemini API

**Prerequisites:**
1. Create a project at [Google AI Studio](https://aistudio.google.com)
2. Generate an API key
3. Install the SDK: `pip install google-generativeai`
4. Set the environment variable: `export GEMINI_API_KEY='your-key-here'`

In [ ]:
# NOTE: Set your API key before running this cell:
#   export GEMINI_API_KEY='your-key-here'
# Or set it programmatically (not recommended for shared notebooks):
#   os.environ["GEMINI_API_KEY"] = "your-key-here"

api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    raise EnvironmentError(
        "GEMINI_API_KEY environment variable not set. "
        "Please set it before running this notebook."
    )

genai.configure(api_key=api_key)
print("Gemini API configured successfully.")

In [6]:
# List all available Gemini models
print("Available models:")
print("-" * 60)

for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(f"  Name       : {model.name}")
        print(f"  Display    : {model.display_name}")
        print(f"  Description: {model.description[:80]}..." if len(model.description) > 80 else f"  Description: {model.description}")
        print()

Available models:
------------------------------------------------------------
  Name       : models/gemini-2.5-flash
  Display    : Gemini 2.5 Flash
  Description: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports ...

  Name       : models/gemini-2.5-pro
  Display    : Gemini 2.5 Pro
  Description: Stable release (June 17th, 2025) of Gemini 2.5 Pro

  Name       : models/gemini-2.0-flash
  Display    : Gemini 2.0 Flash
  Description: Gemini 2.0 Flash

  Name       : models/gemini-2.0-flash-001
  Display    : Gemini 2.0 Flash 001
  Description: Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for ...

  Name       : models/gemini-2.0-flash-exp-image-generation
  Display    : Gemini 2.0 Flash (Image Generation) Experimental
  Description: Gemini 2.0 Flash (Image Generation) Experimental

  Name       : models/gemini-2.0-flash-lite-001
  Display    : Gemini 2.0 Flash-Lite 001
  Description: Stable version of Gemini 2.0 Flash-Lite


In [7]:
# Inspect a specific model's properties
model_info = genai.get_model("models/gemini-2.0-flash")

print(f"Model Name          : {model_info.name}")
print(f"Display Name        : {model_info.display_name}")
print(f"Input Token Limit   : {model_info.input_token_limit:,}")
print(f"Output Token Limit  : {model_info.output_token_limit:,}")
print(f"Supported Methods   : {model_info.supported_generation_methods}")
print(f"Temperature Range   : {model_info.temperature} (default)")
print(f"TopP                : {model_info.top_p}")
print(f"TopK                : {model_info.top_k}")

Model Name          : models/gemini-2.0-flash
Display Name        : Gemini 2.0 Flash
Input Token Limit   : 1,048,576
Output Token Limit  : 8,192
Supported Methods   : ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Temperature Range   : 1.0 (default)
TopP                : 0.95
TopK                : 40


## First API Call: Simple Text Generation

In [8]:
# Create a model instance and generate a simple response
model = genai.GenerativeModel("gemini-2.5-flash")

prompt = "Explain the concept of attention mechanisms in neural networks in 3 sentences."

response = model.generate_content(prompt)

print("Prompt:")
print(f"  {prompt}")
print()
print("Response:")
print(response.text)

Prompt:
  Explain the concept of attention mechanisms in neural networks in 3 sentences.

Response:
Attention mechanisms allow neural networks to dynamically weigh the importance of different parts of the input data when processing information. Instead of processing an entire sequence uniformly, the model "attends" to relevant elements by computing a set of weights that determine how much each input component contributes to the current output or prediction. This selective focus greatly improves performance on tasks involving long sequences, like machine translation or text summarization, by enabling the network to selectively retrieve and integrate contextually important information.


## Generation Configuration

The `GenerationConfig` object controls how the model samples its output:

| Parameter | Range | Effect |
|---|---|---|
| `temperature` | 0.0 – 2.0 | Higher = more creative/random; lower = more deterministic |
| `top_p` | 0.0 – 1.0 | Nucleus sampling: restrict to top-p probability mass |
| `top_k` | 1 – 40 | Restrict sampling to top-k most likely tokens |
| `max_output_tokens` | 1 – 8192 | Maximum tokens in the response |
| `stop_sequences` | list of str | Stop generation when one of these strings is produced |

In [9]:
# Using GenerationConfig to control sampling behavior
generation_config = genai.GenerationConfig(
    temperature=0.2,          # Low temperature for factual, consistent output
    top_p=0.9,                # Nucleus sampling
    top_k=40,                 # Restrict to top 40 tokens
    max_output_tokens=512,    # Limit response length
    stop_sequences=["##"],    # Stop if the model produces '##'
)

model_configured = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config=generation_config,
)

response = model_configured.generate_content(
    "List the top 5 use cases for large language models in enterprise settings. "
    "Format as a numbered list with a one-sentence explanation for each."
)

print(response.text)
print()
print("--- Usage Metadata ---")
print(f"Prompt tokens    : {response.usage_metadata.prompt_token_count}")
print(f"Candidates tokens: {response.usage_metadata.candidates_token_count}")
print(f"Total tokens     : {response.usage_metadata.total_token_count}")

Here are the top 5 use cases for large language models in enterprise settings:

1.

--- Usage Metadata ---
Prompt tokens    : 30
Candidates tokens: 19
Total tokens     : 538


## Safety Settings

Gemini includes built-in safety filters that can be adjusted per harm category:

| Category | Description |
|---|---|
| `HARM_CATEGORY_HARASSMENT` | Bullying, threatening, or abusive content |
| `HARM_CATEGORY_HATE_SPEECH` | Content targeting identity characteristics |
| `HARM_CATEGORY_SEXUALLY_EXPLICIT` | Explicit sexual content |
| `HARM_CATEGORY_DANGEROUS_CONTENT` | Content promoting dangerous activities |

**Block thresholds** (from most to least restrictive):
`BLOCK_LOW_AND_ABOVE` > `BLOCK_MEDIUM_AND_ABOVE` > `BLOCK_ONLY_HIGH` > `BLOCK_NONE`

In [10]:
# Configuring safety settings

safety_settings = {
    HarmCategory.HARM_CATEGORY_HARASSMENT:        HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH:       HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
}

model_safe = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    safety_settings=safety_settings,
)

response = model_safe.generate_content("What are the main risks of deploying AI in healthcare?")

# Check safety ratings
print("Safety Ratings:")
for rating in response.candidates[0].safety_ratings:
    print(f"  {rating.category.name}: {rating.probability.name}")
print()
print("Response:")
print(response.text)


Safety Ratings:

Response:
Deploying Artificial Intelligence (AI) in healthcare holds immense promise, but it also introduces several significant risks that need careful consideration and mitigation. These risks can broadly be categorized as follows:

1.  **Accuracy and Reliability Issues (Misdiagnosis & Treatment Errors):**
    *   **Risk:** AI models, especially in their current state, are not infallible. Errors in their algorithms, flawed training data, or limitations in understanding complex human biology can lead to incorrect diagnoses, inappropriate treatment recommendations, or missed critical conditions.
    *   **Impact:** Patient harm, delayed or incorrect treatment, increased healthcare costs, and erosion of trust in both AI and medical professionals.

2.  **Bias and Fairness (Exacerbating Health Disparities):**
    *   **Risk:** AI models are trained on historical data. If this data is unrepresentative (e.g., disproportionately featuring certain demographics, ethnicities, o

## Multi-turn Conversations (Chat)

The Gemini SDK supports stateful multi-turn conversations via `start_chat()`.

In [11]:
# Multi-turn chat example
model = genai.GenerativeModel("gemini-2.5-flash")
chat = model.start_chat(history=[])

turns = [
    "I'm learning about neural networks. Can you briefly explain what a neuron does?",
    "How does that relate to the transformer architecture?",
    "And what makes Gemini different from earlier transformers?",
]

for user_message in turns:
    print(f"USER: {user_message}")
    response = chat.send_message(user_message)
    print(f"GEMINI: {response.text}")
    print("-" * 60)

print(f"\nTotal turns in history: {len(chat.history)}")

USER: I'm learning about neural networks. Can you briefly explain what a neuron does?
GEMINI: Imagine a neuron in a neural network as a tiny, individual decision-making unit. Its primary job is to **receive inputs, process them, and then produce an output**.

Here's a breakdown of what it does, step-by-step:

1.  **Receives Inputs:** A neuron takes in numerical values from other neurons or directly from the initial data.
2.  **Applies Weights:** Each input value is multiplied by a corresponding **weight**. These weights represent the "strength" or "importance" of that particular input to the neuron's decision. (Larger weight = more influence).
3.  **Sums Inputs (Weighted Sum):** All the weighted inputs are then added together.
4.  **Adds Bias:** A single numerical value called **bias** is added to this sum. The bias allows the neuron to activate even if all its inputs are zero, or to make it harder to activate, effectively shifting the activation threshold.
5.  **Activates (Activation 

# 3. Overview of Vertex AI

## What is Vertex AI?

**Vertex AI** is Google Cloud's unified machine learning platform. It provides end-to-end ML capabilities from data preparation and model training to deployment, monitoring, and MLOps — all under one roof.

For Gemini users, Vertex AI provides an **enterprise-grade access path** to the same Gemini models, with additional benefits:

### Key Services

| Service | Description |
|---|---|
| **Generative AI on Vertex AI** | Access Gemini and other foundation models (Llama, Claude, Mistral) |
| **Model Garden** | Browse, customize, and deploy 150+ foundation models |
| **Vertex AI Studio** | Web-based prompt playground and fine-tuning UI |
| **Training** | Custom model training with AutoML or custom containers |
| **Pipelines** | Orchestrate ML workflows with Kubeflow-based pipelines |
| **Feature Store** | Centralized feature management for ML |
| **Model Registry** | Version, track, and manage models |
| **Endpoints** | Deploy models as low-latency REST endpoints |
| **Evaluation** | Automated model evaluation and benchmarking |
| **Grounding** | Connect model outputs to Google Search or your data |

## Direct API vs. Vertex AI

| Feature | Google AI (Direct API) | Vertex AI |
|---|---|---|
| **Authentication** | API Key | Google Cloud IAM (OAuth2 / Service Account) |
| **Pricing** | Pay-per-token | Pay-per-token + GCP billing integration |
| **Rate Limits** | Generous free tier | Quota configurable per project |
| **Data Residency** | Limited | Choose GCP region |
| **VPC / Private Networking** | No | Yes |
| **CMEK Encryption** | No | Yes |
| **Model Fine-tuning** | Limited | Full support |
| **Grounding (Search/RAG)** | Basic | Advanced (Google Search + data stores) |
| **Audit Logging** | No | Cloud Audit Logs |
| **SLA** | No | Yes (enterprise SLA) |
| **Best For** | Prototyping, personal projects | Production, regulated industries |

## Setting Up Vertex AI

**Prerequisites:**
1. Google Cloud project with billing enabled
2. Enable the Vertex AI API: `gcloud services enable aiplatform.googleapis.com`
3. Authenticate: `gcloud auth application-default login`
4. Install the SDK: `pip install google-cloud-aiplatform`
5. Set environment variables:
   ```bash
   export GCP_PROJECT='your-project-id'
   export GCP_LOCATION='us-central1'
   ```

In [12]:
# NOTE: Set your GCP project before running:
#   export GCP_PROJECT='your-gcp-project-id'
# Authenticate via:
#   gcloud auth application-default login

GCP_PROJECT  = os.environ.get("GCP_PROJECT")
GCP_LOCATION = os.environ.get("GCP_LOCATION", "us-central1")

if not GCP_PROJECT:
    raise EnvironmentError(
        "GCP_PROJECT environment variable not set. "
        "Please set it to your Google Cloud project ID."
    )

# Initialize the Vertex AI SDK
vertexai.init(project=GCP_PROJECT, location=GCP_LOCATION)

print(f"Vertex AI initialized.")
print(f"  Project  : {GCP_PROJECT}")
print(f"  Location : {GCP_LOCATION}")

OSError: GCP_PROJECT environment variable not set. Please set it to your Google Cloud project ID.

In [ ]:
# Using the Vertex AI GenerativeModel
vertex_model = GenerativeModel("gemini-2.5-flash")

vertex_response = vertex_model.generate_content(
    "What are the three most important considerations when deploying LLMs in production?"
)

print("Response via Vertex AI:")
print(vertex_response.text)

In [ ]:
# Generation config with Vertex AI
vertex_config = GenerationConfig(
    temperature=0.3,
    top_p=0.95,
    top_k=40,
    max_output_tokens=1024,
)

vertex_model_configured = GenerativeModel(
    "gemini-2.5-flash",
    generation_config=vertex_config,
)

structured_prompt = """You are an expert ML engineer. 
Provide a concise checklist (5 items) for productionizing a generative AI application.
Format: numbered list, one line per item."""

vertex_response = vertex_model_configured.generate_content(structured_prompt)

print("Structured response via Vertex AI (with config):")
print(vertex_response.text)
print()
print("--- Usage Metadata ---")
print(vertex_response.usage_metadata)

In [ ]:
# System instructions: Vertex AI supports system-level persona
system_instruction_model = GenerativeModel(
    "gemini-2.5-flash",
    system_instruction="""You are a senior data scientist at a Fortune 500 company.
    You communicate in a concise, technical, and results-focused manner.
    Always back up your statements with data or industry best practices."""
)

response = system_instruction_model.generate_content(
    "What is the most effective way to evaluate an LLM for a customer support use case?"
)

print("Response with system instruction:")
print(response.text)

## Comparing API Calls: Direct API vs Vertex AI

Both paths use nearly identical Python interfaces — the main difference is initialization and authentication:

**Direct API (google-generativeai):**
```python
import google.generativeai as genai

genai.configure(api_key="YOUR_API_KEY")
model = genai.GenerativeModel("gemini-2.0-flash")
response = model.generate_content("Hello, Gemini!")
print(response.text)
```

**Vertex AI (google-cloud-aiplatform):**
```python
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project="your-project", location="us-central1")
model = GenerativeModel("gemini-2.0-flash")
response = model.generate_content("Hello, Gemini!")
print(response.text)
```

The core API surface (`generate_content`, `GenerationConfig`, `start_chat`) is essentially the same in both SDKs, making migration between the two straightforward.

In [ ]:
# Side-by-side token counting: both APIs expose count_tokens

test_prompt = "Explain what Vertex AI is in one paragraph."

# Direct API
direct_model = genai.GenerativeModel("gemini-2.5-flash")
direct_count = direct_model.count_tokens(test_prompt)
print(f"Direct API - Token count for test prompt: {direct_count.total_tokens}")

# Vertex AI
vertex_count = vertex_model.count_tokens(test_prompt)
print(f"Vertex AI  - Token count for test prompt: {vertex_count.total_tokens}")

## Summary

In this segment we covered:

1. **Generative AI Foundations** — What generative AI is, the transformer architecture that powers modern LLMs, and the key milestones from 2017 to today.

2. **Gemini API** — The Gemini model family, how to authenticate and configure the `google-generativeai` SDK, make text generation calls, tune generation parameters, manage safety settings, and conduct multi-turn conversations.

3. **Vertex AI** — What Vertex AI is, its key services, how it differs from direct API access, and how to initialize and use the `vertexai` SDK to call Gemini models in an enterprise-ready manner.

### Next Steps

- **Segment 2**: Multimodal inputs — images, audio, video with Gemini
- **Segment 3**: Advanced prompting strategies and structured outputs
- **Segment 4**: RAG, grounding, and tool use / function calling
- **Segment 5**: Building production applications with Vertex AI

<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>